In [ ]:
import pickle
import seaborn
import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data

Main Goal of notebook:\
For each condition (uncertainty, prediction_fixed, prediction_rolling, true_label)\
plot the development of the top-k channels across trials\
the update_linear functions assigns points to the top-k channels per trials, with the top channel getting k points, the second top channel getting k-1 points etc. A minimum of 0 points is awareded. The scores are accumulated across trials.\
The update_same function assiangs the top-k channels in a trial the same number of points and the points are not accumulated.\
The advantage over aggregation over the trial dimension this way is that each trial is of equal importance, which is not given when we use the mean to aggreagte over the trial dimension.\
After inspecting the channel importance across trials with both update_functions, we see if we can determine statistical patterns between uncertainty in a trial and and the top-k channels.\
From the plots it looks like there is a higher variance of channel scores in high variance trials. This can be somewhat statistically represented, if the right number of channels (k) is chosen and a good metric of variation (e.g. std, iqr etc.) is chosen.


## Boilerplate

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [2,4,7,9,11,13,14,18,22,24]
 #test_subject_indices: [2,4,7,9,11,13,14,18,22,24]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


In [ ]:
def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += k-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict, k):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict



In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict, k)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]

    return top_k_channels, channel_point_dicts_individual_trials


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:
cwd = os.getcwd()

cwd = os.getcwd()
file_path = os.path.join(cwd, "../data/subject_007_preprocessed_combined_py.fif")
epochs = mne.read_epochs(file_path)
info = epochs.info
ch_names = epochs.ch_names

In [ ]:
trial_numbers = []
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"

    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)

    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    trial_numbers.append(all_epochs.shape[0])

del all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep,


subjects_dicts = []
subject_indices = [2,4,7,9,11,13,14,22,24]
for i in subject_indices:
    with open(f'./agg_dict_subject_{i}.pickle', 'rb') as handle:
        aggregate_dict = pickle.load(handle)
        subjects_dicts.append(aggregate_dict)


last_indices = []
for i in range(len(subjects_dicts)):
    end_idx = np.where(subjects_dicts[i]["dl_agg"]==0)[0][0]
    last_indices.append(end_idx)

for i in range(len(subjects_dicts)):
    for idx,(k,v) in enumerate(subjects_dicts[i].items()):
        subjects_dicts[i][k] = np.array(v[:last_indices[i]])

Consider adding very high uncertainty condition here

In [ ]:
groupby_dicts_list = []
for i in range(len(subject_indices)):
    
    sigma2 = np.exp(subjects_dicts[i]["pred_uncertainty"])
    # throw away top 5% of values, hoping to remove outliers
    sigma2_indices = sigma2<=np.percentile(sigma2, 95)
    outlier_indices = ~sigma2_indices
    #hist_values.append(sigma2[i][vars_lower_x_percent_bool])   

    # changed levels of thresholds
    low_threshold = np.percentile(sigma2[sigma2_indices], 33)
    high_threshold = np.percentile(sigma2[sigma2_indices], 66)

    low_indices = sigma2 < low_threshold
    medium_indices = (low_threshold<=sigma2) & (sigma2<=high_threshold)
    high_indices = (sigma2>high_threshold)^outlier_indices
    uncertainty_levels = {"low":low_indices, "medium":medium_indices, "high":high_indices}


    binary_label_fixed_zero_bool = subjects_dicts[i]["pred_binary_label_fixed"] == 0
    binary_label_fixed_one_bool = subjects_dicts[i]["pred_binary_label_fixed"] ==  1
    binary_label_fixed_dict = {"pred_zero" : binary_label_fixed_zero_bool, "pred_one" : binary_label_fixed_one_bool}

    binary_label_rolling_zero_bool = subjects_dicts[i]["pred_binary_label_rolling"] == 0
    binary_label_rolling_one_bool = subjects_dicts[i]["pred_binary_label_rolling"] == 1
    binary_label_rolling_dict = {"pred_zero" : binary_label_rolling_zero_bool , "pred_one" : binary_label_rolling_one_bool}

    binary_label_true_zero_bool = subjects_dicts[i]["true_binary_label"] ==  0 
    binary_label_true_one_bool = subjects_dicts[i]["true_binary_label"] ==  1 
    binary_label_true_dict = {"true_zero" : binary_label_true_zero_bool , "true_one" : binary_label_true_one_bool}
    
    groupby_dicts_list.append((uncertainty_levels,binary_label_fixed_dict, binary_label_rolling_dict, binary_label_true_dict))

    

In [ ]:
# get metric for comparing top-k channels
top_k_per_subject = []
for i in range(len(subject_indices)):
    top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(subjects_dicts[i], "dl_agg", ch_names, 10, update_channel_points_linear)
    top_k_per_subject.append((top_k_linear_weighted, points_progression_linear_weighted))

In [ ]:
for i in range(len(subject_indices)):
    print(top_k_per_subject[i][0])

In [ ]:
top_k_per_subject = []
for i in range(len(subject_indices)):
    top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(subjects_dicts[i], "oc_ch_agg", ch_names, 10, update_channel_points_linear)
    top_k_per_subject.append((top_k_linear_weighted, points_progression_linear_weighted))

In [ ]:
for i in range(len(subject_indices)):
    print(top_k_per_subject[i][0])

## groupby uncertainty

In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 20, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15,5), sharey=True)
    fig.tight_layout()
    max_val = 0
    min_val = 0
    for j, groupby_key in enumerate(points_progression_per_subject_uncertainty[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_uncertainty[i][groupby_key][1], ch_names)

        if np.max(list(point_progression_dict.values()))>max_val:
            max_val = np.max(list(point_progression_dict.values()))
            #print(max_val)
        if np.min(list(point_progression_dict.values()))<min_val:
            min_val = np.min(list(point_progression_dict.values()))
            #print(min_val)

        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: uncertainty", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    
    for ax in axs:
        ax.collections[0].set_clim(min_val, max_val)
    dir_path = f"{cwd}/all_subjects/top_k/uncertainty/linear/heatmaps"
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)


    fig.savefig(dir_path+"/Subject_{subject_indices[i]}_uncertainty.png")




In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual )))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_uncertainty[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_uncertainty[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: uncertainty", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/uncertainty/same/heatmaps/Subject_{subject_indices[i]}_uncertainty.png")


## groupby prediction fixed

In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear)))

In [ ]:
subjects_dicts[i]

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    min_val = 0
    max_val = 0
    for j, groupby_key in enumerate(points_progression_per_subject_pred_fixed[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_fixed[i][groupby_key][1], ch_names)

        if np.max(list(point_progression_dict.values()))>max_val:
            max_val = np.max(list(point_progression_dict.values()))
            #print(max_val)
        if np.min(list(point_progression_dict.values()))<min_val:
            min_val = np.min(list(point_progression_dict.values()))
            #print(min_val)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction fixed", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    for ax in axs:
        ax.collections[0].set_clim(min_val, max_val)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_fixed/linear/heatmaps/Subject_{subject_indices[i]}.png")



In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_fixed[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_fixed[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction fixed", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_fixed/same/heatmaps/Subject_{subject_indices[i]}.png")

## groupby prediction rolling

In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    min_val = 0
    max_val = 0

    for j, groupby_key in enumerate(points_progression_per_subject_pred_rolling[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_rolling[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction rolling", y=-0.05)
        axs[j].set_title(groupby_key)

        if np.max(list(point_progression_dict.values()))>max_val:
            max_val = np.max(list(point_progression_dict.values()))
            #print(max_val)
        if np.min(list(point_progression_dict.values()))<min_val:
            min_val = np.min(list(point_progression_dict.values()))
            #print(min_val)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    for ax in axs:
        ax.collections[0].set_clim(min_val, max_val)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_rolling/linear/heatmaps/Subject_{subject_indices[i]}.png")


In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_rolling[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_rolling[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction rolling", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_rolling/same/heatmaps/Subject_{subject_indices[i]}.png")


## groupby true label

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()

    min_val = 0
    max_val = 0
    for j, groupby_key in enumerate(points_progression_per_subject_true[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_true[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction true", y=-0.05)
        axs[j].set_title(groupby_key)
        
        if np.max(list(point_progression_dict.values()))>max_val:
            max_val = np.max(list(point_progression_dict.values()))
            #print(max_val)
        if np.min(list(point_progression_dict.values()))<min_val:
            min_val = np.min(list(point_progression_dict.values()))
            #print(min_val)

        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    for ax in axs:
        ax.collections[0].set_clim(min_val, max_val)
    fig.savefig(f"{cwd}/all_subjects/top_k/true_binary/linear/heatmaps/Subject_{subject_indices[i]}.png")

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_true[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_true[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction true", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/true_binary/same/heatmaps/Subject_{subject_indices[i]}.png")

### This was due to a bug that was fixed!
In the later trials the top-k channels always are the same for some subjects(This was due to a bug that was fixed). Why is this the case? Also note that the subjects for which this is the case are exactly the ones which have a multimodal distribution of uncertainties? Check if the uncertanties start collapsing at later trials for that subject. If that is the case there is likely are relation between an uncertainty output of 1 for a trial and always the same channels being in the top-k for a trial

## top-k stats

Get top-k channels for different conditioning methods, see if top-k differ between conditions and also whether a trend can be discerned for different across subjects which channels are deemed importance. Would be cool if we had some locality information for this (we could see if the top channels are at least close to each other). But this is currently not the case.
Maybe we can define a neighborhood of an EEG grid ourselves and test it like that?

REMEBER: RUnning the following only makes sense with the linear aggreation. not with same! (would only make sense with same if we sum over the trial dimension, which can be tried. This would essentially mean awarding the top-k channel in a trial the same amount of points and then summing over all trials )

In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_channel_points_linear)))

In [ ]:
from scipy.stats import iqr, median_abs_deviation

In [ ]:
# for variation function, possible only measure variance of points for top-k channels?
def variation_per_subject(point_progression_dict, subject_idx, only_top_k=False, variation_measure=np.std, **kwargs):
    variation_per_group = {}

    for groupby_key in point_progression_dict[subject_idx].keys():
        if only_top_k==False:
            variation_per_group[groupby_key] = np.round(variation_measure(list(point_progression_dict[subject_idx][groupby_key][1][-1].values()), **kwargs),2)
        else:
            top_k_points = []
            top_k_channels = point_progression_dict[subject_idx][groupby_key][0]
            for ch in top_k_channels:
                top_k_points.append(point_progression_dict[subject_idx][groupby_key][1][-1].get(ch))
                
            variation_per_group[groupby_key] = np.round(variation_measure(top_k_points),2)

    return variation_per_group

In [ ]:
def variation(point_progression_dict, only_top_k=False, variation_measure=np.std, **kwargs):
    variations_per_subject_list = []
    for i in range(len(point_progression_dict)):
        variations_per_subject_list.append(variation_per_subject(point_progression_dict, i, only_top_k=only_top_k, variation_measure=variation_measure, **kwargs))
    return variations_per_subject_list


### for uncertainty

In [ ]:
variation(points_progression_per_subject_uncertainty, only_top_k=True), variation(points_progression_per_subject_uncertainty)

Interestingly, and something I did not previously consider if (addtionally to removing 5% of outliers) we get the top 20 channels instead of the top 10, the results of both variation options (onlt_top_k =True/False) change. \


In [ ]:
variation(points_progression_per_subject_uncertainty, only_top_k=True, variation_measure=iqr), variation(points_progression_per_subject_uncertainty, variation_measure=iqr)

THe top-20 method did not seem to good for this variation_metric. However for the only_top_k=False condition, the variation of the low condition was consitently lower than the variation of the high condition.

In [ ]:
variation(points_progression_per_subject_uncertainty, only_top_k=True, variation_measure=median_abs_deviation), variation(points_progression_per_subject_uncertainty, variation_measure=median_abs_deviation)

Increasing the top-10 to top-20 seems to have done nothing for this variatio metric. In general it seems to perform the worst out of all available ones so far.

General Notes about experiments:

Increasing the size of the medium group (s.t. it covered 66 percent after removal of outliers) only lead to higher variation in the medium group compared to the other two across options.
Therefore it seems better to keep the old group sizes, at least for making connections between variation and uncertainty, which I was trying to do here

Increasing the size of top-k from k=10 to k=20 seems to often give the high uncertainty channels a higher variation that at least the low uncertainty channels. This seems to work especially well if we look at the variation between all channels!

### for pred fixed

In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear)))

In [ ]:
variation(points_progression_per_subject_pred_fixed, only_top_k=True), variation(points_progression_per_subject_pred_fixed)

This worked well for both the top-k(=10) version and the all channels version, with differences arguably being more pronounced ub tge top-k version.\
The variation is consitently and clearly higher in the pred_one condition

In [ ]:
variation(points_progression_per_subject_pred_fixed, only_top_k=True, variation_measure=iqr), variation(points_progression_per_subject_pred_fixed, variation_measure=iqr)

In [ ]:
variation(points_progression_per_subject_pred_fixed, only_top_k=True, variation_measure=median_abs_deviation), variation(points_progression_per_subject_pred_fixed, variation_measure=median_abs_deviation)

### for pred rolling


In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_linear)))

In [ ]:
variation(points_progression_per_subject_pred_rolling, only_top_k=True), variation(points_progression_per_subject_pred_rolling)

In [ ]:
variation(points_progression_per_subject_pred_rolling, only_top_k=True, variation_measure=iqr), variation(points_progression_per_subject_pred_rolling, variation_measure=iqr)

In [ ]:
variation(points_progression_per_subject_pred_rolling, only_top_k=True, variation_measure=median_abs_deviation), variation(points_progression_per_subject_pred_rolling, variation_measure=median_abs_deviation)

Not consitent statistical pattern can be uncovered here regardless of the metric of variation

### for true label

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_linear)))

In [ ]:
variation(points_progression_per_subject_true, only_top_k=True), variation(points_progression_per_subject_true)

just like for predicted label fixed, this works perfetly gain on both, top-k channels and all channels.\
the true_one condition has consistenly higher std than the true_zero conidtion

In [ ]:
variation(points_progression_per_subject_true, only_top_k=True, variation_measure=iqr), variation(points_progression_per_subject_true, variation_measure=iqr)

In [ ]:
variation(points_progression_per_subject_true, only_top_k=True, variation_measure=median_abs_deviation), variation(points_progression_per_subject_true, variation_measure=median_abs_deviation)

### remarks Top k stats

For the predicted label fixed and true label condition a clear statistical pattern can be determined across subjects.\
That is that the label=1 groups have consitently higher standard deviation than the label=0 groups .even though this might seem counterintuitive looking at the plots above.\
However, since this pattern did not show itself on the other measues of variation that are less outlier-sensitive (iqr and mad), it seems that these differences are mostly caused by a few values being far away from the mean rather than a consistenly higher variation across channels in the label=1 group\
Looking at the heatmaps for the label=1 groups vs label=0 groups seems to reinforce that notion